# EDA 010: User review-mix counts

## Key Goal

Count users in each dataset split (`train`, `val`) by review mix: positive-only, negative-only, and mixed positive+negative; also show how many are multi-review users.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from cwd={here}")

REPO_ROOT = _repo_root()
PROCESSED = REPO_ROOT / "data" / "processed"
TRAIN_PARQUET = PROCESSED / "steam_reviews_cleaned_english_train_norm.parquet"
VAL_PARQUET = PROCESSED / "steam_reviews_cleaned_english_val_norm.parquet"

for path in (TRAIN_PARQUET, VAL_PARQUET):
    if not path.is_file():
        raise FileNotFoundError(path)

USER_COL = "author.steamid"
USECOLS = [USER_COL, "review_id", "app_id", "recommended"]


In [2]:
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_parquet(path, columns=USECOLS).copy()
    df["split"] = split_name
    return df

train_df = load_split(TRAIN_PARQUET, "train")
val_df = load_split(VAL_PARQUET, "val")

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
display(pd.DataFrame({
    "split": ["train", "val"],
    "rows": [len(train_df), len(val_df)],
    "unique_users": [train_df[USER_COL].nunique(), val_df[USER_COL].nunique()],
}))


Train rows: 6410755
Val rows: 1374737


,split,rows,unique_users
0,train,6410755,3998456
1,val,1374737,1172653


In [3]:
def summarize_user_mix(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    user_summary = (
        df.groupby(USER_COL)
        .agg(
            n_reviews=("review_id", "nunique"),
            n_apps=("app_id", "nunique"),
            n_positive=("recommended", lambda s: int((s == 1).sum())),
            n_negative=("recommended", lambda s: int((s == 0).sum())),
        )
        .reset_index()
    )

    user_summary["review_mix"] = "mixed_positive_negative"
    user_summary.loc[user_summary["n_negative"] == 0, "review_mix"] = "positive_only"
    user_summary.loc[user_summary["n_positive"] == 0, "review_mix"] = "negative_only"
    user_summary["is_multi_review"] = user_summary["n_apps"] >= 2

    counts = (
        user_summary.groupby("review_mix")
        .agg(
            users=(USER_COL, "count"),
            multi_review_users=("is_multi_review", "sum"),
            single_review_users=("is_multi_review", lambda s: int((~s).sum())),
            median_reviews=("n_reviews", "median"),
            median_unique_games=("n_apps", "median"),
        )
        .reset_index()
    )
    counts["user_share_pct"] = 100.0 * counts["users"] / counts["users"].sum()
    counts["multi_review_share_within_group_pct"] = 100.0 * counts["multi_review_users"] / counts["users"]
    return user_summary, counts.sort_values("users", ascending=False, ignore_index=True)

train_user_summary, train_counts = summarize_user_mix(train_df)
val_user_summary, val_counts = summarize_user_mix(val_df)


In [4]:
print("Train split: users by review mix")
display(train_counts)

print("Val split: users by review mix")
display(val_counts)


Train split: users by review mix


,review_mix,users,multi_review_users,single_review_users,median_reviews,median_unique_games,user_share_pct,multi_review_share_within_group_pct
0,positive_only,3416458,871219,2545239,1.0,1.0,85.444432,25.500650
1,negative_only,315298,24285,291013,1.0,1.0,7.885494,7.702237
2,mixed_positive_negative,266700,266700,0,3.0,3.0,6.670075,100.000000


Val split: users by review mix


,review_mix,users,multi_review_users,single_review_users,median_reviews,median_unique_games,user_share_pct,multi_review_share_within_group_pct
0,positive_only,1027762,113044,914718,1.0,1.0,87.644171,10.999045
1,negative_only,113585,4527,109058,1.0,1.0,9.686156,3.985561
2,mixed_positive_negative,31306,31306,0,2.0,2.0,2.669673,100.000000


In [5]:
overall = pd.DataFrame([
    {
        "split": "train",
        "users": len(train_user_summary),
        "multi_review_users_ge2_games": int(train_user_summary["is_multi_review"].sum()),
        "multi_review_share_pct": 100.0 * float(train_user_summary["is_multi_review"].mean()),
    },
    {
        "split": "val",
        "users": len(val_user_summary),
        "multi_review_users_ge2_games": int(val_user_summary["is_multi_review"].sum()),
        "multi_review_share_pct": 100.0 * float(val_user_summary["is_multi_review"].mean()),
    },
])
print("Overall multi-review counts by split")
display(overall)


Overall multi-review counts by split


,split,users,multi_review_users_ge2_games,multi_review_share_pct
0,train,3998456,1162204,29.066320
1,val,1172653,148877,12.695742


In [6]:
train_users = set(train_user_summary[USER_COL].tolist())
val_users = set(val_user_summary[USER_COL].tolist())
users_in_both = train_users & val_users
all_users = train_users | val_users

overlap_summary = pd.DataFrame([
    {
        "train_users": len(train_users),
        "val_users": len(val_users),
        "users_in_both_train_and_val": len(users_in_both),
        "share_of_train_users_pct": 100.0 * len(users_in_both) / len(train_users),
        "share_of_val_users_pct": 100.0 * len(users_in_both) / len(val_users),
        "share_of_union_users_pct": 100.0 * len(users_in_both) / len(all_users),
    }
])

print("User overlap between train and val")
display(overlap_summary)


User overlap between train and val


,train_users,val_users,users_in_both_train_and_val,share_of_train_users_pct,share_of_val_users_pct,share_of_union_users_pct
0,3998456,1172653,592853,14.827048,50.556559,12.94932


In [7]:
# Optional: inspect user-level rows for one group
train_user_summary[train_user_summary["review_mix"] == "mixed_positive_negative"].head(20)


,author.steamid,n_reviews,n_apps,n_positive,n_negative,review_mix,is_multi_review
22,76561197960267039,2,2,1,1,mixed_positive_negative,True
29,76561197960267699,2,2,1,1,mixed_positive_negative,True
35,76561197960267972,2,2,1,1,mixed_positive_negative,True
57,76561197960269110,2,2,1,1,mixed_positive_negative,True
59,76561197960269155,5,5,2,3,mixed_positive_negative,True
65,76561197960269230,5,5,3,2,mixed_positive_negative,True
66,76561197960269234,2,2,1,1,mixed_positive_negative,True
77,76561197960269425,2,2,1,1,mixed_positive_negative,True
78,76561197960269447,3,3,1,2,mixed_positive_negative,True
92,76561197960269775,5,5,3,2,mixed_positive_negative,True
